# Experiment 12: Sample complexity, PAC learning, and VC dimension

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import learning_curve
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, y = load_digits(return_X_y=True)
sizes, train_scores, val_scores = learning_curve(
    make_pipeline(StandardScaler(), SVC()), X, y,
    train_sizes=np.linspace(0.05, 1.0, 10), cv=5)

plt.plot(sizes, val_scores.mean(axis=1), marker="o", label="validation")
plt.plot(sizes, train_scores.mean(axis=1), marker="o", label="training")
plt.fill_between(sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                 val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.2)
plt.xlabel("training set size")
plt.ylabel("accuracy")
plt.title("Learning curve: performance vs training size")
plt.legend()
plt.show()

In [ ]:
# PAC bound for a finite hypothesis class: m >= (1/eps) * (ln|H| + ln(1/delta))
import math
H_size = 1000
for eps, delta in [(0.1, 0.05), (0.05, 0.05), (0.1, 0.01)]:
    m = math.ceil((1 / eps) * (math.log(H_size) + math.log(1 / delta)))
    print(f"|H|={H_size}, epsilon={eps}, delta={delta}  ->  m >= {m} samples")

In [ ]:
# Toy PAC experiment: learn the threshold t in h(x) = 1[x >= t] for x uniform in [0, 1]
rng = np.random.default_rng(0)
true_t = 0.6
trials = 200
sample_sizes = [5, 10, 20, 50, 100, 500]
avg_err = []
for m in sample_sizes:
    errs = []
    for _ in range(trials):
        x = rng.uniform(size=m)
        labels = (x >= true_t).astype(int)
        pos = x[labels == 1]
        t_hat = pos.min() if len(pos) else 1.0    # consistent hypothesis
        errs.append(abs(t_hat - true_t))          # true error = probability mass between thresholds
    avg_err.append(np.mean(errs))

plt.plot(sample_sizes, avg_err, marker="o")
plt.xscale("log")
plt.xlabel("number of samples m")
plt.ylabel("average true error")
plt.title("PAC toy example: error decreases with more samples")
plt.show()

In [ ]:
# VC dimension of linear classifiers in 2D is 3: 3 points can be shattered, 4 cannot
import itertools
from sklearn.svm import SVC

def shattered(points):
    points = np.array(points, dtype=float)
    for labels in itertools.product([0, 1], repeat=len(points)):
        if len(set(labels)) < 2:
            continue   # all-same labels are always achievable
        clf = SVC(kernel="linear", C=1e6).fit(points, labels)
        if clf.score(points, labels) < 1.0:
            return False
    return True

three = [[0, 0], [1, 0], [0, 1]]
four_xor = [[0, 0], [1, 1], [1, 0], [0, 1]]
print("3 points shattered by linear classifiers:", shattered(three))
print("4 XOR-style points shattered:", shattered(four_xor))